In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from numpy.lib.stride_tricks import sliding_window_view

keras.utils.set_random_seed(42)
DATA = Path("data")

## Building blocks, forward and backward

In [2]:
def conv2d(X, K):
    """Valid convolution of one image X (h x w) with filter K (f x f)."""
    return (sliding_window_view(X, K.shape) * K).sum(axis=(-1, -2))

def maxpool(A):
    h, w = A.shape[0] // 2, A.shape[1] // 2
    return A[:2 * h, :2 * w].reshape(h, 2, w, 2).max(axis=(1, 3))

def maxpool_backward(dP, A):
    """Send each gradient to the position of its window's maximum (the first one if tied); others get 0."""
    h, w = dP.shape
    blocks = A[:2 * h, :2 * w].reshape(h, 2, w, 2).transpose(0, 2, 1, 3).reshape(h, w, 4)
    mask = np.eye(4)[blocks.argmax(axis=-1)]                          # 1 at the maximum of each window
    routed = np.where(mask == 1, dP[..., None], 0.0).reshape(h, w, 2, 2).transpose(0, 2, 1, 3).reshape(2 * h, 2 * w)
    dA = np.zeros_like(A)
    dA[:2 * h, :2 * w] = routed
    return dA

def forward(X, W1, b1, W2, b2):
    Z1 = conv2d(X, W1) + b1
    A1 = np.maximum(Z1, 0)
    P1 = maxpool(A1)
    F = P1.reshape(-1, 1)
    A2 = 1 / (1 + np.exp(-(W2 @ F + b2)))
    return Z1, A1, P1, F, A2

def backward(X, t, W1, b1, W2, b2):
    Z1, A1, P1, F, A2 = forward(X, W1, b1, W2, b2)
    dZ2 = A2 - t                                     # dL/dZ2 = a2 - y (part 1)
    dW2, db2 = dZ2 @ F.T, dZ2.item()
    dF = W2.T * dZ2                                  # dL/dF = W2^T (a2 - y)
    dP1 = dF.reshape(P1.shape)                       # flatten backward: reshape
    dA1 = maxpool_backward(dP1, A1)                  # max pooling backward: route
    dZ1 = dA1 * (Z1 > 0)                             # ReLU backward: mask
    db1 = dZ1.sum()                                  # bias: sum of dL/dZ1
    dW1 = conv2d(X, dZ1)                             # filter: convolve X with dL/dZ1
    return dict(W1=dW1, b1=db1, W2=dW2, b2=db2), dict(dP1=dP1, dA1=dA1, dZ1=dZ1, A1=A1)

## The worked example: max pooling backward on a 4 x 4 map

In [3]:
A1 = np.array([[1, 5, 2, 3], [2, 4, 0, 1], [7, 1, 4, 2], [3, 0, 1, 3]], float)
dP1 = np.array([[0.1, -0.2], [0.3, 0.4]])
print("maxpool:\n", maxpool(A1))
print("routed gradient:\n", maxpool_backward(dP1, A1))

maxpool:
 [[5. 3.]
 [7. 4.]]
routed gradient:
 [[ 0.   0.1  0.  -0.2]
 [ 0.   0.   0.   0. ]
 [ 0.3  0.   0.4  0. ]
 [ 0.   0.   0.   0. ]]


## The worked example: a 3 x 3 input and a 2 x 2 filter
dL/dW1 must equal the convolution of X with dL/dZ1; dL/db1 the sum of dL/dZ1.

In [4]:
X = np.arange(1, 10, dtype=float).reshape(3, 3)
W = np.array([[0.2, -0.1], [0.4, 0.3]])
G = np.array([[0.5, -1.0], [0.25, 2.0]])          # pretend dL/dZ1 coming from above
Xv, Wv = tf.Variable(X), tf.Variable(W)
bv = tf.Variable(0.0, dtype=tf.float64)
with tf.GradientTape() as tape:
    Z = tf.nn.conv2d(Xv[None, :, :, None], Wv[:, :, None, None], 1, "VALID")[0, :, :, 0] + bv
    L = tf.reduce_sum(Z * G)                       # a loss whose dL/dZ is exactly G
gW, gb = tape.gradient(L, [Wv, bv])
print("conv(X, dL/dZ1):\n", conv2d(X, G))
print("TensorFlow dL/dW1:\n", gW.numpy())
print("sum of dL/dZ1:", G.sum(), "  TensorFlow dL/db1:", gb.numpy())

conv(X, dL/dZ1):
 [[ 9.5  11.25]
 [14.75 16.5 ]]
TensorFlow dL/dW1:
 [[ 9.5  11.25]
 [14.75 16.5 ]]
sum of dL/dZ1: 1.75   TensorFlow dL/db1: 1.75


## The full network on a real digit, against GradientTape
MNIST 28 x 28: conv 3 x 3 -> 26 x 26 -> ReLU -> max pool -> 13 x 13 -> flatten 169 -> sigmoid.
Task: is the digit a 1 (y = 1) or a 7 (y = 0)?

In [5]:
(xtr, ytr), (xte, yte) = keras.datasets.mnist.load_data()
def pick(x, y):
    keep = np.isin(y, [1, 7])
    return x[keep] / 255.0, (y[keep] == 1).astype(float)
xtr, ytr = pick(xtr, ytr)
xte, yte = pick(xte, yte)
print("train", xtr.shape, " test", xte.shape)

rng = np.random.default_rng(0)
params = dict(W1=rng.normal(0, 0.3, (3, 3)), b1=0.0, W2=rng.normal(0, 0.1, (1, 169)), b2=0.0)
grads, inner = backward(xtr[0], ytr[0], **params)

v = {k: tf.Variable(np.array(val, float)) for k, val in params.items()}
with tf.GradientTape() as tape:
    Z1 = tf.nn.conv2d(tf.constant(xtr[0])[None, :, :, None], v["W1"][:, :, None, None], 1, "VALID")[0, :, :, 0] + v["b1"]
    P1 = tf.nn.max_pool2d(tf.nn.relu(Z1)[None, :, :, None], 2, 2, "VALID")[0, :, :, 0]
    a2 = tf.sigmoid(v["W2"] @ tf.reshape(P1, (-1, 1)) + v["b2"])
    L = -(ytr[0] * tf.math.log(a2) + (1 - ytr[0]) * tf.math.log(1 - a2))
tg = tape.gradient(L, list(v.values()))
for (k, mine), theirs in zip(grads.items(), tg):
    print(f"{k}: largest difference from TensorFlow {np.abs(np.array(mine) - theirs.numpy()).max():.2e}")
print("dL/dW1 by hand:\n", grads["W1"].round(5))
print("nonzero entries of dL/dA1:", (inner["dA1"] != 0).sum(), "of", inner["dA1"].size, "(at most one per pooling window: 169)")

train (13007, 28, 28)  test (2163, 28, 28)


W1: largest difference from TensorFlow 4.86e-17
b1: largest difference from TensorFlow 0.00e+00
W2: largest difference from TensorFlow 5.55e-17
b2: largest difference from TensorFlow 0.00e+00
dL/dW1 by hand:
 [[-0.07475 -0.02015 -0.06614]
 [ 0.01424  0.01666  0.01015]
 [ 0.02616  0.01167 -0.02107]]
nonzero entries of dL/dA1: 169 of 676 (at most one per pooling window: 169)


## Training with these formulas only
Mini-batch gradient descent, batch 32, learning rate 0.1, 2 epochs, gradients averaged over the batch.

In [6]:
def accuracy(p, X, Y):
    preds = np.array([forward(x, **p)[-1].item() for x in X])
    return ((preds > 0.5) == Y).mean()

lr, batch = 0.1, 32
log = []
step = 0
print("test accuracy before training:", accuracy(params, xte, yte).round(4))
for epoch in range(2):
    order = rng.permutation(len(xtr))
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        total = {k: 0.0 for k in params}
        loss = 0.0
        for i in idx:
            g, _ = backward(xtr[i], ytr[i], **params)
            a2 = forward(xtr[i], **params)[-1].item()
            loss += -(ytr[i] * np.log(a2 + 1e-12) + (1 - ytr[i]) * np.log(1 - a2 + 1e-12))
            for k in params:
                total[k] = total[k] + g[k]
        for k in params:                                   # gradient descent: w <- w - lr * dL/dw
            params[k] = params[k] - lr * total[k] / len(idx)
        if step % 20 == 0:
            log.append(dict(step=step, loss=loss / len(idx)))
        step += 1
    print(f"epoch {epoch + 1}: test accuracy {accuracy(params, xte, yte):.4f}")
pd.DataFrame(log).to_csv(DATA / "numpy_training.csv", index=False)
print("learned filter:\n", params["W1"].round(2))
np.save(DATA / "learned_filter.npy", params["W1"])

test accuracy before training: 0.5469


epoch 1: test accuracy 0.9898


epoch 2: test accuracy 0.9940
learned filter:
 [[0.5  0.4  0.46]
 [0.65 0.49 0.57]
 [1.   0.88 0.22]]


## The 6 x 6 digit of part 1, backward step by step
The same image, filter and weights as the part 1 Notebook (MNIST 0s and 1s shrunk to 6 x 6, weights from `default_rng(0)`). Every intermediate gradient is saved for the figure.

In [7]:
from PIL import Image
(x6, y6), _ = keras.datasets.mnist.load_data()
keep6 = np.isin(y6, [0, 1])
X6 = np.array(Image.fromarray(x6[keep6][0]).resize((6, 6), Image.BILINEAR)) / 255.0
t6 = float(y6[keep6][0] == 1)
rng6 = np.random.default_rng(0)
p6 = dict(W1=rng6.normal(0, 0.5, (3, 3)), b1=0.1, W2=rng6.normal(0, 0.5, (1, 4)), b2=0.0)
g6, inner6 = backward(X6, t6, **p6)
Z1_6 = conv2d(X6, p6["W1"]) + p6["b1"]
print("a2 - y =", round(g6["b2"], 4), "(part 1: 0.2104)")
print("dL/dP1:\n", inner6["dP1"].round(4))
print("dL/dA1:\n", inner6["dA1"].round(4))
print("Z1 > 0 everywhere:", bool((Z1_6 > 0).all()))
print("dL/db1 =", round(g6["b1"], 4))
print("dL/dW1:\n", g6["W1"].round(4))
np.savez(DATA / "digit_backward.npz", X=X6, t=t6, Z1=Z1_6, **{k: np.asarray(v) for k, v in inner6.items()},
         dW1=g6["W1"], db1=g6["b1"], W2=p6["W2"], dZ2=g6["b2"])

a2 - y = 0.2104 (part 1: 0.2104)
dL/dP1:
 [[-0.1331 -0.0656]
 [ 0.0043 -0.2445]]
dL/dA1:
 [[ 0.      0.      0.     -0.0656]
 [ 0.     -0.1331  0.      0.    ]
 [ 0.      0.0043 -0.2445  0.    ]
 [ 0.      0.      0.      0.    ]]
Z1 > 0 everywhere: True
dL/db1 = -0.4388
dL/dW1:
 [[-0.1231 -0.0958 -0.2059]
 [-0.1007 -0.1183 -0.1286]
 [-0.1839 -0.1362 -0.0404]]


In [8]:
# Worked numbers quoted in the Note (sections 4, 7 and 9), at five decimals so the rounding matches
np.set_printoptions(precision=5, suppress=True)
print("W2 =", p6["W2"].round(5), "  a2 - y =", round(g6["b2"], 5))
print("dL/dF = W2^T (a2 - y):", (p6["W2"].ravel() * g6["b2"]).round(5))
dZ1_6 = inner6["dZ1"]
nz = list(zip(*np.nonzero(dZ1_6)))
print("non-zero cells of dL/dZ1:", [(int(r), int(c), round(dZ1_6[r, c], 5)) for r, c in nz])
print("their sum, dL/db1 =", round(dZ1_6.sum(), 5))
print("dL/dW1[0,0] products:", [(round(dZ1_6[r, c], 5), round(X6[r, c], 5), round(dZ1_6[r, c] * X6[r, c], 5)) for r, c in nz],
      " sum", round(g6["W1"][0, 0], 5))
# ReLU backward on a small example: dL/dA and Z
dA_s = np.array([[0.1, -0.2], [0.3, 0.4]]); Z_s = np.array([[1.5, -0.7], [2.0, -0.1]])
print("mask:\n", (Z_s > 0).astype(float), "\ndL/dZ = dL/dA * mask:\n", dA_s * (Z_s > 0))

W2 = [[-0.63271 -0.31164  0.02066 -1.16252]]   a2 - y = 0.21036
dL/dF = W2^T (a2 - y): [-0.1331  -0.06556  0.00435 -0.24454]
non-zero cells of dL/dZ1: [(0, 3, np.float64(-0.06556)), (1, 1, np.float64(-0.1331)), (2, 1, np.float64(0.00435)), (2, 2, np.float64(-0.24454))]
their sum, dL/db1 = -0.43885
dL/dW1[0,0] products: [(np.float64(-0.06556), np.float64(0.13333), np.float64(-0.00874)), (np.float64(-0.1331), np.float64(0.00784), np.float64(-0.00104)), (np.float64(0.00435), np.float64(0.17647), np.float64(0.00077)), (np.float64(-0.24454), np.float64(0.46667), np.float64(-0.11412))]  sum -0.12314
mask:
 [[1. 0.]
 [1. 0.]] 
dL/dZ = dL/dA * mask:
 [[ 0.1 -0. ]
 [ 0.3  0. ]]
